# AML Compound-Pattern Detection ? IBM HI-Small

This notebook implements the methodology agreed in the linked chat: build a temporal, directed account graph; establish an edge-aware message-passing baseline; mine bounded AML compound-pattern (ACP) candidates from training history; screen candidates by support and lift; freeze a compact library; then combine pattern and graph representations with hierarchical attention. The main compute bottleneck is bounded pattern extraction, so this is a capped prototype rather than a full 5M-transaction run.

Use `HI-Small_Trans.csv` and `HI-Small_accounts.csv` initially. Do not use `HI-Small_Patterns.txt` to design or train ACPs; reserve it for later pattern-recovery validation. Evaluation uses chronological periods and reports PR-AUC, ROC-AUC, F1, and Precision/Recall@K. The first target below is an account alert proxy (whether its outgoing transactions in a day include a laundering label); it is not yet prospective future-horizon risk prediction.

In [1]:
from pathlib import Path
import math, time
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve
import torch
from torch import nn

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path.cwd()
DATA_DIR = ROOT / "Data" if (ROOT / "Data").exists() else ROOT.parent / "Data"
TRANS_PATH = DATA_DIR / "HI-Small_Trans.csv"
ACCOUNTS_PATH = DATA_DIR / "HI-Small_accounts.csv"
assert TRANS_PATH.exists(), f"Missing {TRANS_PATH}"
assert ACCOUNTS_PATH.exists(), f"Missing {ACCOUNTS_PATH}"
print("Device:", DEVICE, "Data:", DATA_DIR.resolve())

Device: cpu Data: D:\AntiMoneyLaundering_FF_detection\Data


## 1. Load HI-Small
The CSV has two columns named `Account`; explicit names preserve sender and receiver separately. Limit memory with deterministic time-stratified sampling so train, validation, and test all span the date range.

In [2]:
COLS = ["Timestamp", "From Bank", "From Account", "To Bank", "To Account", "Amount Received", "Receiving Currency", "Amount Paid", "Payment Currency", "Payment Format", "Is Laundering"]
tx = pd.read_csv(TRANS_PATH, header=0, names=COLS, parse_dates=["Timestamp"],
    dtype={"From Bank":"string", "From Account":"string", "To Bank":"string", "To Account":"string", "Payment Format":"category", "Is Laundering":"int8"})
accounts = pd.read_csv(ACCOUNTS_PATH, dtype={"Bank ID":"string", "Account Number":"string", "Entity ID":"string"})
tx["Amount Paid"] = pd.to_numeric(tx["Amount Paid"], errors="coerce").fillna(0).clip(lower=0)
tx = tx.dropna(subset=["Timestamp", "From Account", "To Account"]).sort_values("Timestamp", kind="stable").reset_index(drop=True)
# Prototype cap spread across days; set None to use all rows (requires substantial RAM/time).
MAX_ROWS = 250_000
if MAX_ROWS and len(tx) > MAX_ROWS:
    tx["_day"] = tx.Timestamp.dt.floor("D")
    total = len(tx)
    keep = []
    for _, g in tx.groupby("_day", sort=True, observed=True):
        n = max(1, round(MAX_ROWS * len(g) / total))
        keep.extend(g.index[np.linspace(0, len(g)-1, min(n, len(g))).astype(int)].tolist())
    tx = tx.loc[sorted(set(keep))].drop(columns="_day").reset_index(drop=True)
tx["log_amount"] = np.log1p(tx["Amount Paid"].astype("float32"))
tx["from_key"] = tx["From Bank"].astype(str) + ":" + tx["From Account"].astype(str)
tx["to_key"] = tx["To Bank"].astype(str) + ":" + tx["To Account"].astype(str)
tx["time_block"] = tx.Timestamp.dt.floor("D")
print(f"Rows: {len(tx):,}; dates: {tx.Timestamp.min()} to {tx.Timestamp.max()}; laundering tx: {int(tx['Is Laundering'].sum()):,}")
display(tx.head(), accounts.head())

Rows: 249,999; dates: 2022-09-01 00:00:00 to 2022-09-18 01:18:00; laundering tx: 242


,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,log_amount,from_key,to_key,time_block
0,2022-09-01,03209,8000F4670,03209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0,9.594007,03209:8000F4670,03209:8000F4670,2022-09-01
1,2022-09-01,001,8001AADD0,001,8001AADD0,145397.31,US Dollar,145397.31,US Dollar,Reinvestment,0,11.887232,001:8001AADD0,001:8001AADD0,2022-09-01
2,2022-09-01,01362,800392D50,01362,800392D50,55538.73,US Dollar,55538.73,US Dollar,Reinvestment,0,10.924854,01362:800392D50,01362:800392D50,2022-09-01
3,2022-09-01,01729,80043AE10,01729,80043AE10,646472.01,US Dollar,646472.01,US Dollar,Reinvestment,0,13.379287,01729:80043AE10,01729:80043AE10,2022-09-01
4,2022-09-01,00795,8004E4160,001244,80198DAE0,6.26,US Dollar,6.26,US Dollar,Credit Card,0,1.982380,00795:8004E4160,001244:80198DAE0,2022-09-01


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


## 2. Chronological split and stable graph IDs
Use early time blocks for training, then validation, then future test. Account nodes are bank-account pairs. Account-to-entity mapping is kept for later analysis; account identity is used for the initial transaction graph.

In [3]:
nodes = pd.Index(pd.concat([tx.from_key, tx.to_key], ignore_index=True).unique())
node_id = {key:i for i,key in enumerate(nodes)}
tx["src"] = tx.from_key.map(node_id).astype("int32")
tx["dst"] = tx.to_key.map(node_id).astype("int32")
rel_names = sorted(tx["Payment Format"].astype("string").fillna("UNKNOWN").unique())
rel_id = {name:i for i,name in enumerate(rel_names)}
tx["rel"] = tx["Payment Format"].astype("string").fillna("UNKNOWN").map(rel_id).astype("int16")
blocks = np.array(sorted(tx.time_block.unique()))
if len(blocks) < 5: raise ValueError("Need at least 5 daily blocks for chronological train/validation/test.")
a = max(1, int(len(blocks)*.70)); b = max(a+1, int(len(blocks)*.85))
train_end, val_end = blocks[a-1], blocks[min(b-1, len(blocks)-2)]
tx["split"] = np.where(tx.time_block <= train_end, "train", np.where(tx.time_block <= val_end, "validation", "test"))
print(tx.groupby("split", observed=True).agg(rows=("src","size"), positives=("Is Laundering","sum"), start=("Timestamp","min"), end=("Timestamp","max")))
print(f"Nodes={len(nodes):,}; relation types={len(rel_names)}")

              rows  positives               start                 end
split                                                                
test             4          3 2022-09-16 00:01:00 2022-09-18 01:18:00
train       249978        227 2022-09-01 00:00:00 2022-09-12 23:51:00
validation      17         12 2022-09-13 01:08:00 2022-09-15 23:38:00
Nodes=200,715; relation types=7


## 3. Edge-aware message-passing baseline
For each prediction day, aggregate only directed transactions from the preceding 30 days. Messages encode sender embedding, relation/payment format, log amount, and recency. Day labels are account-level maxima over that day's outgoing transaction labels. This simple edge-aware MPNN is the baseline stage; it does not consume ACP features.

In [4]:
HISTORY_DAYS = 30
MAX_HISTORY_EDGES = 100_000
DIM = 32
N = len(nodes); R = len(rel_names)

def block_data(day):
    day = pd.Timestamp(day)
    h = tx[(tx.Timestamp < day) & (tx.Timestamp >= day-pd.Timedelta(days=HISTORY_DAYS))]
    if len(h) > MAX_HISTORY_EDGES: h = h.iloc[-MAX_HISTORY_EDGES:]
    age = ((day-h.Timestamp).dt.total_seconds()/86400).to_numpy(np.float32)
    current = tx[tx.time_block == day].groupby("src")["Is Laundering"].max()
    return (torch.tensor(h.src.to_numpy(np.int64),device=DEVICE), torch.tensor(h.dst.to_numpy(np.int64),device=DEVICE),
            torch.tensor(h.rel.to_numpy(np.int64),device=DEVICE), torch.tensor(h.log_amount.to_numpy(np.float32),device=DEVICE),
            torch.tensor(age,device=DEVICE), torch.tensor(current.index.to_numpy(np.int64),device=DEVICE),
            torch.tensor(current.to_numpy(np.float32),device=DEVICE))

class EdgeMPNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.node = nn.Embedding(N,DIM); self.relation = nn.Embedding(R,DIM)
        self.edge = nn.Sequential(nn.Linear(2,DIM),nn.ReLU(),nn.Linear(DIM,DIM))
        self.message = nn.Sequential(nn.Linear(3*DIM,DIM),nn.ReLU(),nn.Linear(DIM,DIM))
        self.update = nn.GRUCell(DIM,DIM); self.norm=nn.LayerNorm(DIM)
        self.classifier=nn.Sequential(nn.Linear(DIM,DIM),nn.ReLU(),nn.Linear(DIM,1))
    def encode(self,src,dst,rel,amount,age):
        x=self.node.weight
        if len(src):
            e=self.edge(torch.stack([amount,age],1))
            m=self.message(torch.cat([x[src],self.relation(rel),e],1))
            agg=torch.zeros_like(x); agg.index_add_(0,dst,m)
            deg=torch.zeros((N,1),device=DEVICE); deg.index_add_(0,dst,torch.ones((len(dst),1),device=DEVICE))
            x=self.norm(self.update(agg/deg.clamp_min(1),x))
        return x
    def forward(self,data):
        src,dst,rel,amt,age,target,_=data
        h=self.encode(src,dst,rel,amt,age)
        return h,self.classifier(h[target]).squeeze(-1)

baseline=EdgeMPNN().to(DEVICE)
print(f"Baseline parameters: {sum(p.numel() for p in baseline.parameters()):,}")

Baseline parameters: 6,435,905


## 4. Bounded two-hop ACP candidates (dominant compute cost)
Mine candidates only on the training prefix. A candidate is a directed path A?B?C with relation pair, elapsed-time bucket, and log-amount-difference bucket. Apply a 7-day temporal window, maximum edge sample, per-account branching cap, and total-instance cap. Complexity of unconstrained neighborhood search grows roughly with branching^h; the caps are essential.

In [ ]:
MAX_PATTERN_EDGES = 100_000
BRANCH_CAP = 10
MAX_INSTANCES = 300_000
MAX_SPAN = pd.Timedelta(days=7)

def extract_candidates(edges):
    e=edges.sort_values("Timestamp",kind="stable").iloc[:MAX_PATTERN_EDGES]
    out={}
    # Plain tuples preserve access to column names containing spaces.
    columns = ["src", "dst", "Timestamp", "rel", "log_amount", "Is Laundering"]
    for src, dst, timestamp, rel, log_amount, label in e[columns].itertuples(index=False, name=None):
        out.setdefault(int(src),[]).append((int(dst),timestamp,int(rel),float(log_amount),int(label)))
    out={u:es[:BRANCH_CAP] for u,es in out.items()}
    rows=[]
    for a,es in out.items():
        for b,t1,r1,v1,y1 in es:
            for c,t2,r2,v2,y2 in out.get(b,[]):
                if t2<t1 or t2-t1>MAX_SPAN: continue
                gap=int((t2-t1).total_seconds()//3600)
                delta=round(v2-v1,1)
                rows.append((a,b,c,r1,r2,gap,delta,max(y1,y2),t2))
                if len(rows)>=MAX_INSTANCES: break
            if len(rows)>=MAX_INSTANCES: break
        if len(rows)>=MAX_INSTANCES: break
    return pd.DataFrame(rows,columns=["src","mid","dst","rel1","rel2","gap_h","delta","label","event_time"])

train_edges=tx[tx.split=="train"][["Timestamp","src","dst","rel","log_amount","Is Laundering"]].copy()
t0=time.time(); instances=extract_candidates(train_edges)
print(f"Bounded candidate instances: {len(instances):,} in {time.time()-t0:.1f}s")
display(instances.head())

## 5. Frequency/lift screen and freeze pattern library
After structural candidate generation, calculate training-only enrichment against transaction labels. Apply minimum support and lift, then freeze the retained ACP signatures. Labels from the withheld pattern annotation file are not consulted.

In [ ]:
MIN_SUPPORT=20; MIN_LIFT=1.2; MAX_ACPS=24
if instances.empty: raise RuntimeError("No candidates extracted; inspect sampling/window settings.")
SIG=["rel1","rel2","gap_h","delta"]
instances["pattern_id"]=instances.groupby(SIG,sort=False,observed=True).ngroup()
base_rate=instances.label.mean()
screen=(instances.groupby("pattern_id",observed=True).agg(support=("label","size"),positive_rate=("label","mean")).reset_index())
screen["lift"]=screen.positive_rate/max(base_rate,1e-12)
screen=screen[(screen.support>=MIN_SUPPORT)&(screen.lift>=MIN_LIFT)].sort_values(["lift","support"],ascending=False).head(MAX_ACPS).reset_index(drop=True)
ACP_IDS=screen.pattern_id.astype(int).tolist()
ACP_INDEX={p:i for i,p in enumerate(ACP_IDS)}
SIG_TO_ID={tuple([int(r.rel1),int(r.rel2),int(r.gap_h),float(r.delta)]):int(r.pattern_id) for r in instances.drop_duplicates("pattern_id").itertuples()}
print(f"Training instance base rate={base_rate:.4%}; frozen ACP library={len(ACP_IDS)}")
display(screen.head(15))

## 6. Hierarchical attention, fusion, and classifier
Within each ACP family, attention pools motif instances for an account (inner attention). Across retained ACP families, a second attention layer weights each family (cross-ACP attention). Fuse the ACP vector with the edge-MPNN account embedding and classify. Attention is a ranking aid, not a standalone explanation.

In [ ]:
class ACPModel(nn.Module):
    def __init__(self,k):
        super().__init__(); self.k=max(1,k)
        self.instance=nn.Sequential(nn.Linear(4,DIM),nn.ReLU(),nn.Linear(DIM,DIM))
        self.inner=nn.Linear(DIM,1); self.cross=nn.Linear(DIM,1)
        self.fuse=nn.Sequential(nn.Linear(2*DIM,DIM),nn.ReLU(),nn.Dropout(.2),nn.Linear(DIM,1))
    def forward(self,h,feats,inst_nodes,inst_pattern,target):
        out=h.new_zeros((len(target),self.k,DIM))
        mapping=torch.full((N,),-1,dtype=torch.long,device=DEVICE); mapping[target]=torch.arange(len(target),device=DEVICE)
        if len(feats):
            z=self.instance(feats); g=mapping[inst_nodes]; valid=g>=0; z=z[valid]; g=g[valid]; p=inst_pattern[valid]
            group=g*self.k+p; score=self.inner(z).squeeze(-1)
            mx=torch.full((len(target)*self.k,),-1e9,device=DEVICE); mx.scatter_reduce_(0,group,score,reduce="amax",include_self=True)
            w=torch.exp(score-mx[group]); den=torch.zeros_like(mx); den.index_add_(0,group,w); w=w/den[group].clamp_min(1e-12)
            out.view(-1,DIM).index_add_(0,group,z*w[:,None])
        mask=out.abs().sum(-1)>0; scores=self.cross(out).squeeze(-1).masked_fill(~mask,-1e9)
        beta=torch.softmax(scores,1)*mask.any(1,keepdim=True)
        pattern=(out*beta[:,:,None]).sum(1)
        return self.fuse(torch.cat([h[target],pattern],1)).squeeze(-1),beta

acp_model=ACPModel(len(ACP_IDS)).to(DEVICE)
print("ACP model ready; families:",len(ACP_IDS))

## 7. Match frozen ACPs in pre-cutoff history
For each target day, search bounded historical paths and attach each matching motif to its participating target accounts. Histories are strictly prior to the target day; after training, discovery is frozen and validation/test only match that library.

In [ ]:
MAX_DAY_INSTANCES=20_000

def pattern_batch(day,target):
    day=pd.Timestamp(day)
    hist=tx[(tx.Timestamp<day)&(tx.Timestamp>=day-pd.Timedelta(days=HISTORY_DAYS))]
    # Library was discovered on train; validation/test match only this frozen training library.
    if day>pd.Timestamp(train_end): hist=hist[hist.Timestamp<=pd.Timestamp(train_end)+pd.Timedelta(days=1)]
    hist=hist.iloc[-MAX_HISTORY_EDGES:]
    out={}
    for r in hist.sort_values("Timestamp").itertuples(index=False):
        out.setdefault(int(r.src),[]).append((int(r.dst),r.Timestamp,int(r.rel),float(r.log_amount)))
    out={u:es[:BRANCH_CAP] for u,es in out.items()}
    targets=set(map(int,target)); rows=[]
    for a,es in out.items():
        for b,t1,r1,v1 in es:
            for c,t2,r2,v2 in out.get(b,[]):
                if t2<t1 or t2-t1>MAX_SPAN: continue
                gap=int((t2-t1).total_seconds()//3600); delta=round(v2-v1,1)
                pid=SIG_TO_ID.get((r1,r2,gap,float(delta)))
                if pid not in ACP_INDEX: continue
                for n in (a,b,c):
                    if n in targets: rows.append((n,ACP_INDEX[pid],v1,v2,gap/168.0,float(r1+r2)))
                    if len(rows)>=MAX_DAY_INSTANCES: break
                if len(rows)>=MAX_DAY_INSTANCES: break
            if len(rows)>=MAX_DAY_INSTANCES: break
        if len(rows)>=MAX_DAY_INSTANCES: break
    if not rows:
        return (torch.empty((0,4),device=DEVICE),torch.empty(0,dtype=torch.long,device=DEVICE),torch.empty(0,dtype=torch.long,device=DEVICE))
    arr=np.asarray(rows,np.float32)
    return (torch.tensor(arr[:,2:6],device=DEVICE),torch.tensor(arr[:,0],dtype=torch.long,device=DEVICE),torch.tensor(arr[:,1],dtype=torch.long,device=DEVICE))

def fit_epoch(days,model,optimizer,with_acp=False):
    model.train(); acp_model.train(); losses=[]
    for day in days:
        data=block_data(day); y=data[-1]
        if not len(y): continue
        optimizer.zero_grad(set_to_none=True)
        h=model.encode(*data[:5]); logits=model.classifier(h[data[5]]).squeeze(-1)
        if with_acp and ACP_IDS:
            feats,ns,ps=pattern_batch(day,data[5].detach().cpu().numpy())
            logits,_=acp_model(h,feats,ns,ps,data[5])
        pos=y.sum(); pw=((len(y)-pos)/pos.clamp_min(1)).clamp(1,100)
        loss=nn.functional.binary_cross_entropy_with_logits(logits,y,pos_weight=pw)
        loss.backward(); torch.nn.utils.clip_grad_norm_(list(model.parameters())+(list(acp_model.parameters()) if with_acp else []),1.0); optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return np.mean(losses) if losses else np.nan

train_days=[d for d in blocks if tx.loc[tx.time_block==d,"split"].iloc[0]=="train"]
val_days=[d for d in blocks if tx.loc[tx.time_block==d,"split"].iloc[0]=="validation"]
test_days=[d for d in blocks if tx.loc[tx.time_block==d,"split"].iloc[0]=="test"]
print("Days train/validation/test:",len(train_days),len(val_days),len(test_days))

## 8. Train in computational order
First train the less expensive graph baseline, then train the fused ACP model. The dominant cost remains ACP extraction/matching, not attention. Epoch counts are intentionally small for this local prototype.

In [ ]:
def score_days(days,model,with_acp=False):
    model.eval(); acp_model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for day in days:
            data=block_data(day); h=model.encode(*data[:5]); logits=model.classifier(h[data[5]]).squeeze(-1)
            if with_acp and ACP_IDS:
                feats,ns,pat=pattern_batch(day,data[5].cpu().numpy()); logits,_=acp_model(h,feats,ns,pat,data[5])
            ys.extend(data[-1].cpu().numpy()); ps.extend(torch.sigmoid(logits).cpu().numpy())
    return np.asarray(ys),np.asarray(ps)

BASE_EPOCHS=2
opt=torch.optim.AdamW(baseline.parameters(),lr=1e-3,weight_decay=1e-4)
for ep in range(BASE_EPOCHS): print("Baseline",ep+1,"loss",fit_epoch(train_days,baseline,opt,False))

fused=EdgeMPNN().to(DEVICE); fused.load_state_dict(baseline.state_dict())
acp_model=ACPModel(len(ACP_IDS)).to(DEVICE)
ACP_EPOCHS=2
opt=torch.optim.AdamW(list(fused.parameters())+list(acp_model.parameters()),lr=5e-4,weight_decay=1e-4)
for ep in range(ACP_EPOCHS): print("HMPNN+ACP",ep+1,"loss",fit_epoch(train_days,fused,opt,True))
val_y,val_p=score_days(val_days,fused,True)
if len(np.unique(val_y))>1:
    pr,re,th=precision_recall_curve(val_y,val_p); f=2*pr[:-1]*re[:-1]/np.maximum(pr[:-1]+re[:-1],1e-12)
    THRESHOLD=float(th[np.argmax(f)]) if len(th) else .5
else: THRESHOLD=.5
print("Validation-selected threshold:",THRESHOLD)

## 9. Future-period metrics
Precision@K and Recall@K reflect a limited investigator alert queue. Results must be reported together with the chronological split, sampled data size, and account-alert proxy definition; do not compare these directly to published metrics using a different task/split.

In [ ]:
def metric_row(y,p):
    m={"PR-AUC":average_precision_score(y,p) if len(np.unique(y))>1 else np.nan,
       "ROC-AUC":roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan,
       "F1":f1_score(y,p>=THRESHOLD,zero_division=0),
       "Precision":precision_score(y,p>=THRESHOLD,zero_division=0),
       "Recall":recall_score(y,p>=THRESHOLD,zero_division=0)}
    order=np.argsort(-p)
    for k in (100,500,1000):
        n=min(k,len(y)); m[f"Precision@{k}"]=float(y[order[:n]].mean()) if n else np.nan
        m[f"Recall@{k}"]=float(y[order[:n]].sum()/max(y.sum(),1)) if n else np.nan
    return m

test_y,test_base=score_days(test_days,baseline,False)
test_y2,test_acp=score_days(test_days,fused,True)
assert np.array_equal(test_y,test_y2)
results=pd.DataFrame({"Edge-aware baseline":metric_row(test_y,test_base),"HMPNN + hierarchical ACP":metric_row(test_y,test_acp)}).T
display(results)
print(f"Rows={len(tx):,}; temporal split={train_end} / {val_end} / future test; history={HISTORY_DAYS} days; caps: edges={MAX_PATTERN_EDGES:,}, branches={BRANCH_CAP}, instances={MAX_INSTANCES:,}.")

## 10. Review the fixed ACP library and next steps
Interpret attention together with ablations: remove edge features, temporal gap, inner attention, cross-pattern attention, and compare performance. To test known-pattern recovery, only after the ACP library is frozen, parse `HI-Small_Patterns.txt` as a held-out reference. For final work, use a prospective future label horizon, walk-forward evaluation, multiple seeds, and measured extraction time/memory. Expand beyond two-hop motifs only when the bounded prototype is tractable. Dataset progression from the shared methodology: HI-Small development ? Tide controlled experiments ? TransXion generalization ? Elliptic2 subgraph research.

In [ ]:
display(screen.head(20))
print("Payment-format relation IDs:", rel_id)
# Keep the annotation file out of discovery/training. It is present locally but intentionally not loaded here.